In [1]:
import pandas as pd
import numpy as np

In [2]:
# ---- 1. Load Amazon dataset ----
amazon = pd.read_csv('../data/raw/Amazon-GoogleProducts/Amazon.csv', encoding='latin1')

In [3]:
# Keep only rows with a populated manufacturer field
amazon_clean = amazon[amazon['manufacturer'].notna() & (amazon['manufacturer'].str.strip() != '')].copy()
print(f"Rows with manufacturer populated: {len(amazon_clean)}")

Rows with manufacturer populated: 1363


In [4]:
# ---- 2. Sample 500 rows ----
sample = amazon_clean.sample(500, random_state=42).reset_index(drop=True)

# List of all distinct manufacturers available to swap in
all_manufacturers = amazon_clean['manufacturer'].unique().tolist()

In [5]:
# ---- 3. Randomly decide which rows get corrupted (mislabeled) ----
rng = np.random.default_rng(seed=42)
sample['is_mislabeled'] = rng.integers(0, 2, size=len(sample))  # 0 or 1, ~50/50

In [6]:
# ---- 4. Build the swapped manufacturer field ----
def get_wrong_manufacturer(true_manufacturer, all_manufacturers, rng):
    # Keep picking until we get one that's actually different
    candidates = [m for m in all_manufacturers if m != true_manufacturer]
    return rng.choice(candidates)

manufacturer_shown = []
for _, row in sample.iterrows():
    if row['is_mislabeled'] == 1:
        wrong = get_wrong_manufacturer(row['manufacturer'], all_manufacturers, rng)
        manufacturer_shown.append(wrong)
    else:
        manufacturer_shown.append(row['manufacturer'])

sample['manufacturer_shown'] = manufacturer_shown
sample = sample.rename(columns={'manufacturer': 'true_manufacturer'})

In [7]:
# ---- 5. Build final output ----
output = sample[['id', 'title', 'manufacturer_shown', 'true_manufacturer', 'is_mislabeled']]

print(f"\nMislabeled count: {(output['is_mislabeled'] == 1).sum()}")
print(f"Correct count: {(output['is_mislabeled'] == 0).sum()}")


Mislabeled count: 244
Correct count: 256


In [8]:
# ---- 6. Sanity check: print samples of each ----
pd.set_option('display.max_colwidth', 60)

print("\n--- 5 Mislabeled examples (manufacturer was swapped) ---")
print(output[output['is_mislabeled'] == 1][['title', 'manufacturer_shown', 'true_manufacturer']].head(5))

print("\n--- 5 Correct examples (manufacturer untouched) ---")
print(output[output['is_mislabeled'] == 0][['title', 'manufacturer_shown', 'true_manufacturer']].head(5))


--- 5 Mislabeled examples (manufacturer was swapped) ---
                              title       manufacturer_shown true_manufacturer
1        adobe contribute cs3 [mac]  humongous entertainment             adobe
2        microsoft excel 2004 (mac)        feral interactive         microsoft
5                 photoedit 2 (mac)          software cinema           macware
7          kaspersky anti-virus 6.0       steinberg (yamaha)     kaspersky lab
10  pc cakewalk guitar tracks pro 3              informatics          cakewalk

--- 5 Correct examples (manufacturer untouched) ---
                                                     title  \
0                                   zonealarm anti-spyware   
3   zonealarm firewall pro 5 (free upgrade to new version)   
4                        the guild 2 - pirates of the seas   
6  symc backup exec aofo 11d win advanced open file option   
8         adobe creative suite cs3 design standard upgrade   

         manufacturer_shown         true_m

In [9]:
# ---- 7. Save ----
output.to_csv('../data/processed/amazon_brand_mislabel_pairs.csv', index=False)
print("\nSaved to data/processed/amazon_brand_mislabel_pairs.csv")


Saved to data/processed/amazon_brand_mislabel_pairs.csv
